<a href="https://colab.research.google.com/github/idialloaka-ai/DAILYCHALLENGE/blob/master/ExercisesXP_Heart_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP - Heart Disease Prediction (Student, Hints Only)

## What you will learn
- Load and inspect CSV data
- EDA and preprocessing
- Train Logistic Regression, SVM, XGBoost
- Hyperparameter tuning with GridSearchCV
- Evaluate with standard metrics

## What you will create
- Working classifiers and a simple comparison report


## Setup

In [ ]:
import os, zipfile, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# Hint: install xgboost in Colab if missing
# !pip install xgboost
try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## Exercise 1 - Exploratory Data Analysis

In [ ]:
import os, zipfile, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# Hint: install xgboost in Colab if missing
# !pip install xgboost
try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# TODO: extract the dataset zip to an output folder
ZIP_PATH = 'Heart Disease Prediction Dataset.zip'  # or your path
EXTRACT_DIR = 'heart_ds'
# Hint: use zipfile.ZipFile(ZIP_PATH).extractall(EXTRACT_DIR)
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

# TODO: list CSV files under EXTRACT_DIR
csv_files = glob.glob(os.path.join(EXTRACT_DIR, '*.csv'))
csv_path = csv_files[0]  # Assuming there is only one CSV file

# TODO: load the CSV into a DataFrame named df
df = pd.read_csv(csv_path)

# TODO: inspect df
display(df.head())
display(df.info())
display(df.describe())

# TODO: identify target column
target = 'target'  # 'target'

# TODO: split features and target
X = df.drop(columns=[target])
y = df[target]

# TODO: train test split with stratification
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
# Hint: train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

### Basic visual checks

In [ ]:
import seaborn as sns
# Visual check of the target distribution
plt.figure(figsize=(6, 4))
sns.countplot(x=y)
plt.title('Distribution of Target (Heart Disease)')
plt.show()

# Numeric distributions
X_train.hist(figsize=(12, 10), bins=20)
plt.tight_layout()
plt.show()

## Preprocessing pipeline

In [ ]:
# Identify numeric and categorical columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

# Preprocessing pipeline
pre = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)
print(f"Numerical columns: {num_cols}")
print(f"Categorical columns: {cat_cols}")

## Helper - evaluation function

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

def eval_and_report(name, model, X_test, y_test):
    y_pred = model.predict(X_test)

    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred)
    }

    print(f"--- {name} ---")
    for k, v in metrics.items():
        print(f"{k.capitalize()}: {v:.4f}")

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ConfusionMatrixDisplay.from_estimator(model, X_test, y_test, ax=ax[0], cmap='Blues')

    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        ax[1].plot(fpr, tpr, label=f'AUC: {roc_auc_score(y_test, y_prob):.4f}')
        ax[1].plot([0, 1], [0, 1], 'k--')
        ax[1].set_title('ROC Curve')
        ax[1].legend()

    plt.show()
    return metrics

## Exercise 2 - Logistic Regression without Grid Search

In [ ]:
pipe_lr = Pipeline([
    ('pre', pre),
    ('lr', LogisticRegression(solver='liblinear', max_iter=1000, random_state=RANDOM_STATE))
])

pipe_lr.fit(X_train, y_train)
lr_no_gs_metrics = eval_and_report('Logistic Regression (No GS)', pipe_lr, X_test, y_test)

## Exercise 3 - Logistic Regression with Grid Search

In [ ]:
pipe_lr_cv = Pipeline([
    ('pre', pre),
    ('lr', LogisticRegression(solver='liblinear', max_iter=1000, random_state=RANDOM_STATE))
])

param_grid = {
    'lr__C': [0.01, 0.1, 1, 10, 100],
    'lr__penalty': ['l1', 'l2']
}

grid_lr = GridSearchCV(pipe_lr_cv, param_grid, cv=5, scoring='f1', n_jobs=-1)
grid_lr.fit(X_train, y_train)

print('Best params:', grid_lr.best_params_)
best_lr = grid_lr.best_estimator_
lr_gs_metrics = eval_and_report('Logistic Regression (Grid Search)', best_lr, X_test, y_test)

## Exercise 4 - SVM without Grid Search

In [ ]:
pipe_svm = Pipeline([
    ('pre', pre),
    ('svm', SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=RANDOM_STATE))
])

pipe_svm.fit(X_train, y_train)
svm_no_metrics = eval_and_report('SVM (No GS)', pipe_svm, X_test, y_test)

## Exercise 5 - SVM with Grid Search

In [ ]:
pipe_svm_cv = Pipeline([
    ('pre', pre),
    ('svm', SVC(probability=True, random_state=RANDOM_STATE))
])

svm_param_grid = {
    'svm__kernel': ['rbf', 'linear', 'poly'],
    'svm__C': [0.1, 1, 10],
    'svm__gamma': ['scale', 'auto']
}

grid_svm = GridSearchCV(pipe_svm_cv, svm_param_grid, cv=5, scoring='f1', n_jobs=-1)
grid_svm.fit(X_train, y_train)

print('Best params:', grid_svm.best_params_)
best_svm = grid_svm.best_estimator_
svm_gs_metrics = eval_and_report('SVM (Grid Search)', best_svm, X_test, y_test)

## Exercise 6 - XGBoost without Grid Search

In [ ]:
pipe_xgb = Pipeline([
    ('pre', pre),
    ('xgb', XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=RANDOM_STATE))
])

pipe_xgb.fit(X_train, y_train)
xgb_no_metrics = eval_and_report('XGBoost (No GS)', pipe_xgb, X_test, y_test)

## Exercise 7 - XGBoost with Grid Search

In [ ]:
pipe_xgb_cv = Pipeline([
    ('pre', pre),
    ('xgb', XGBClassifier(random_state=RANDOM_STATE))
])

xgb_param_grid = {
    'xgb__n_estimators': [50, 100, 200],
    'xgb__learning_rate': [0.01, 0.1, 0.2],
    'xgb__max_depth': [3, 5, 7]
}

grid_xgb = GridSearchCV(pipe_xgb_cv, xgb_param_grid, cv=5, scoring='f1', n_jobs=-1)
grid_xgb.fit(X_train, y_train)

print('Best params:', grid_xgb.best_params_)
best_xgb = grid_xgb.best_estimator_
xgb_gs_metrics = eval_and_report('XGBoost (Grid Search)', best_xgb, X_test, y_test)

## Compare models

In [ ]:
summary = {
    'LR (No GS)': lr_no_gs_metrics,
    'LR (Grid Search)': lr_gs_metrics,
    'SVM (No GS)': svm_no_metrics,
    'SVM (Grid Search)': svm_gs_metrics,
    'XGB (No GS)': xgb_no_metrics,
    'XGB (Grid Search)': xgb_gs_metrics
}

summary_df = pd.DataFrame.from_dict(summary, orient='index')
display(summary_df.sort_values(by='f1', ascending=False))